# Container Build & Run Quickstart

This notebook demonstrates how to build and validate a benchmark container for a single repository commit using the refactored Datasmith orchestration APIs.

## Usage
1. Update the repository parameters below.
2. Point `CONTEXT_REGISTRY_PATH` at a registry JSON that includes your repo/sha (or rely on the default context).
3. Run the notebook top-to-bottom.

> **Note:** Docker, `asv`, and the Datasmith Python package must be available in the current environment.

In [ ]:
%load_ext autoreload
%autoreload 2
%cd /mnt/sdd1/atharvas/formulacode/datasmith
import json
from pathlib import Path

import pandas as pd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
/mnt/sdd1/atharvas/formulacode/datasmith
(455, 15)


In [67]:
results_df = pd.DataFrame([
    json.loads(line)
    for line in Path("scratch/artifacts/pipeflush/symbolic_synthesis/results.jsonl").read_text().splitlines()
    if line != "null"
])
errors = results_df.query("not can_install")
print(errors.shape)
offenders = set(errors["dry_run_log"].str.extract(r"Because ([\w\-]+) was not found").dropna().values.flatten())
offenders.update(
    set(errors["dry_run_log"].str.extract(r"Because there are no versions of ([\w\-]+)").dropna().values.flatten())
)
# Because you require pyvisa==1.11.1 and pyvisa>=1.11.3,<1.12.dev0, we can
offenders

(1169, 15)


{'0-29-32',
 '1-0',
 '1-2',
 '1-22-0',
 '1-3-2',
 '2-18-4',
 '2024-1-1',
 '3-0-0a10',
 'absl',
 'afl',
 'allel',
 'cartopy-userconfig',
 'closest-peak-direction-getter',
 'conans',
 'cprofile',
 'dask-core',
 'dateutil',
 'dbe',
 'deepchecks-metrics',
 'geopandas-base',
 'install',
 'interpnd',
 'jpeg-ls',
 'libblas',
 'libpantab',
 'libwriter',
 'mo-pack',
 'mpl-toolkits',
 'pylab',
 'pyqt4',
 'pytables',
 'python',
 'skbuild',
 'sklearnex',
 'skspatial',
 'system',
 'tunits',
 'urllib2',
 'vcr'}

In [63]:
from datasmith.core.models.task import Task

d = errors.groupby("dry_run_log").head(1).to_dict(orient="records")

tasks = [
    Task(
        owner=row["repo_name"].split("/")[0],
        repo=row["repo_name"].split("/")[1],
        sha=row["sha"],
    )
    for row in d
]
print(len(tasks))

417


In [ ]:
from datasmith.execution.resolution import analyze_commit

TASK = tasks[0]
print(TASK)

r = [analyze_commit(sha=t.sha, repo_name=f"{t.owner}/{t.repo}", bypass_cache=True) for t in tasks[:20]]
r

In [4]:
# CONTEXT_REGISTRY_PATH = Path('scratch/artifacts/context_registry_init.json')
OUTPUT_DIR = Path("scratch/notebooks/output").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")

Output directory: /mnt/sdd1/atharvas/formulacode/datasmith/scratch/notebooks/scratch/notebooks/output


In [5]:
import json

from datasmith.docker.context import ContextRegistry, DockerContext
from datasmith.notebooks.utils import merge_registries, update_cr

pkg_task = TASK.with_tag("pkg")

registries = Path("scratch/").rglob("**/*context_registry*.json")
merged_json = merge_registries(list(registries))
registry = update_cr(ContextRegistry.deserialize(payload=json.dumps(merged_json)))


context = registry.get(pkg_task)
context.building_data == DockerContext().building_data

21:20:32 WARNING  simple_useragent.core: Falling back to historic user agent.
21:20:32 WARNING  datasmith: CACHE_LOCATION environment variable not set. Using default 'cache.db'.
21:20:32 INFO     datasmith.docker.context: No context found for key 'Task(owner='pygeos', repo='pygeos', sha='2afdfd3e4251d7a89cb956f64b79a74326c49830', commit_date=0.0, env_payload='', tag='pkg')'. Using default context.


True

In [13]:
# --- Connect to Docker ---
from datasmith.docker.orchestrator import get_docker_client

client = get_docker_client()
client

In [14]:
# --- Build the package image ---
from datasmith.docker.orchestrator import build_repo_sha_image

build_result = build_repo_sha_image(
    client=client,
    docker_ctx=context,
    task=pkg_task,
    run_id="notebook-run",
)
build_result

20:40:58 INFO     datasmith.docker.context: Docker image 'numpy-numpy-financial-3263e718a6cc2d10ae4e3e4ba4d4c7ed41ee12e8:pkg' not found locally. Building.
20:40:58 INFO     datasmith.docker.context: $ docker build -t numpy-numpy-financial-3263e718a6cc2d10ae4e3e4ba4d4c7ed41ee12e8:pkg . --build-arg REPO_URL='https://www.github.com/numpy/numpy-financial' --build-arg COMMIT_SHA='3263e718a6cc2d10ae4e3e4ba4d4c7ed41ee12e8' --build-arg ENV_PAYLOAD=''
20:41:30 ERROR    datasmith.docker.context: Build failed for 'numpy-numpy-financial-3263e718a6cc2d10ae4e3e4ba4d4c7ed41ee12e8:pkg' in 31.4 sec: [The command '/bin/sh -c chmod +x /workspace/repo/docker_build_pkg.sh &&     /workspace/repo/docker_build_pkg.sh' returned a non-zero code: 1][l.py", line 16, in <module>
    import numpy as np
ModuleNotFoundError: No module named 'numpy'
]


BuildResult(ok=False, image_name='numpy-numpy-financial-3263e718a6cc2d10ae4e3e4ba4d4c7ed41ee12e8:pkg', image_id=None, rc=1, duration_s=31.36229920387268, stderr_tail="The command '/bin/sh -c chmod +x /workspace/repo/docker_build_pkg.sh &&     /workspace/repo/docker_build_pkg.sh' returned a non-zero code: 1\n", stdout_tail='uests>=2.30.0->sphinx>=7.0->numpy-financial==2.0.0) (3.4.3)\nRequirement already satisfied: idna<4,>=2.5 in /opt/conda/envs/asv_3.10/lib/python3.10/site-packages (from requests>=2.30.0->sphinx>=7.0->numpy-financial==2.0.0) (3.10)\nRequirement already satisfied: urllib3<3,>=1.21.1 in /opt/conda/envs/asv_3.10/lib/python3.10/site-packages (from requests>=2.30.0->sphinx>=7.0->numpy-financial==2.0.0) (2.5.0)\nRequirement already satisfied: certifi>=2017.4.17 in /opt/conda/envs/asv_3.10/lib/python3.10/site-packages (from requests>=2.30.0->sphinx>=7.0->numpy-financial==2.0.0) (2025.8.3)\nCollecting soupsieve>1.2 (from beautifulsoup4->pydata-sphinx-theme>=0.15->numpy-financi

In [18]:
print(build_result.stdout_tail)

uests>=2.30.0->sphinx>=7.0->numpy-financial==2.0.0) (3.4.3)
  Obtaining dependency information for soupsieve>1.2 from https://files.pythonhosted.org/packages/14/a0/bb38d3b76b8cae341dad93a2dd83ab7462e6dbcdd84d43f54ee60a8dc167/soupsieve-2.8-py3-none-any.whl.metadata
  Obtaining dependency information for execnet>=2.1 from https://files.pythonhosted.org/packages/43/09/2aea36ff60d16dd8879bdb2f5b3ee0ba8d08cbbdcdfe870e695ce3784385/execnet-2.1.1-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.4/587.4 kB 21.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 61.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 81.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 79.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 54.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 58.2 MB/s  0:00:00
  Building editable for numpy-financial (pyproject.toml): started
  Running com

In [ ]:
# --- Run a quick validation profile ---
import asv

from datasmith.docker.validation import DockerValidator, ValidationConfig

raw_defaults = asv.machine.Machine.get_defaults()  # type: ignore[attr-defined]
machine_defaults = {k: str(v).replace(" ", "_") for k, v in raw_defaults.items()}

config = ValidationConfig(output_dir=OUTPUT_DIR)
validator = DockerValidator(
    client=client,
    context_registry=registry,
    machine_defaults=machine_defaults,
    config=config,
)

run_result = validator.validate_task(TASK.with_tag("run"))
run_result

In [ ]:
# --- Optional cleanup ---
from datasmith.docker.orchestrator import generate_run_labels

labels = generate_run_labels(TASK, run_id="notebook-run")
print("Labels for this run:", labels)
print("Use docker CLI to prune images/containers if desired.")